# StopTheCap - E2-C1 three-class generator attribution (Colab, single T4)

Fine-tunes **distilbert-base-cased** on the 11,887 AI rows of
`datasets/canonical/e2c1.jsonl`, predicting which of three generators wrote a
passage: `deepseek_v41_flash`, `muse_spark_1_3` or `longcat_25`.

Human rows are excluded. The loss is class-weighted because LongCat is only
about 4% of the corpus. Reports and `last.pt` checkpoints are written to Drive
so a disconnect can resume.

This notebook does not rebuild data, tune on test rows, or upload anything.


In [ ]:
import os
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/StopTheCap_E2C1.zip"
REPO_DIR = "/content/StopTheCap"
REPORT_DIR = "/content/drive/MyDrive/StopTheCap_reports/E2C1_attribution"
REPORT_PARENT = "/content/drive/MyDrive/StopTheCap_reports"

CORPUS_SHA256 = "767853403bb77d72bbfa8a2a3e5dfd424ddfa5ebe9626e4cea3a6b1ccba36675"
AI_ROWS = 11887
TRAIN_AI_ROWS = 9344
GENERATORS = ("deepseek_v41_flash", "muse_spark_1_3", "longcat_25")
EXPECTED_GENERATOR_COUNTS = {
    "deepseek_v41_flash": 5248,
    "muse_spark_1_3": 6131,
    "longcat_25": 508,
}
EXPECTED_PARTITIONS = {"train": 9344, "val": 1135, "test_holdout": 1408}

MODEL_NAME = "distilbert-base-cased"
MAX_LENGTH = 512
BATCH_SIZE = 16
GRAD_ACCUM = 2
EPOCHS = 3

print("ZIP_PATH   :", ZIP_PATH)
print("REPORT_DIR :", REPORT_DIR)
print("corpus hash:", CORPUS_SHA256)
print("AI rows    :", AI_ROWS)

## 1. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted at /content/drive")

## 2. Load the private project ZIP

In [ ]:
# Changing the runtime type restarts the kernel and wipes the config cell.
if "LOAD_METHOD" not in globals():
    import os
    LOAD_METHOD = "zip"
    ZIP_PATH = "/content/drive/MyDrive/StopTheCap_E2C1.zip"
    REPO_DIR = "/content/StopTheCap"
    print("NOTE: config cell had not run; using default paths.")

import os, shutil, sys, zipfile
from pathlib import Path

REQUIRED = [
    "ml/__init__.py",
    "ml/train/train_attribution.py",
    "ml/ingest/__init__.py",
    "ml/ingest/normalise.py",
    "datasets/canonical/e2c1.jsonl",
    "datasets/manifests/e2_c1_training.json",
    "notebooks/colab_e2c1_attribution.ipynb",
]
EXTRACT_TMP = "/content/_stc_attribution_extract"


def _fail(msg):
    raise RuntimeError(msg)


def load_from_zip(zip_path, dest):
    z = Path(zip_path)
    if not z.is_file():
        _fail(
            "ZIP not found: " + str(z) + "\n"
            "Upload the project ZIP to Google Drive, then set ZIP_PATH.\n"
            "Build it with: python scripts/make_colab_zip.py --with-e2c1-training"
        )
    dest = Path(dest)
    tmp = Path(EXTRACT_TMP)
    for p in (dest, tmp):
        if p.exists():
            shutil.rmtree(p)
    tmp.mkdir(parents=True)

    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
        print("zip entries:", len(names))
        if not names:
            _fail("The ZIP is empty.")
        zf.extractall(tmp)

    entries = [p for p in tmp.iterdir() if p.name != "__MACOSX"]
    root = entries[0] if len(entries) == 1 and entries[0].is_dir() else tmp
    shutil.move(str(root), str(dest))
    if tmp.exists():
        shutil.rmtree(tmp, ignore_errors=True)
    return dest


if LOAD_METHOD == "zip":
    REPO = load_from_zip(ZIP_PATH, REPO_DIR)
else:
    _fail("This private notebook loads the Drive ZIP only; set LOAD_METHOD = 'zip'.")

REPO = Path(REPO).resolve()
missing = [f for f in REQUIRED if not (REPO / f).is_file()]
if missing:
    _fail("Project structure looks wrong; missing: " + ", ".join(missing))

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("project root:", REPO)
print("required    :", len(REQUIRED), "present")

## 3. Validate the corpus hash and AI-only counts

In [ ]:
import hashlib, json
from collections import Counter

CORPUS_PATH = REPO / "datasets/canonical/e2c1.jsonl"
TRAINING_MANIFEST_PATH = REPO / "datasets/manifests/e2_c1_training.json"

digest = hashlib.sha256(CORPUS_PATH.read_bytes()).hexdigest()
assert digest == CORPUS_SHA256, f"corpus sha256 {digest} != pinned {CORPUS_SHA256}"

training_manifest = json.loads(TRAINING_MANIFEST_PATH.read_text(encoding="utf-8"))
assert training_manifest["corpus_sha256"] == CORPUS_SHA256

rows = [json.loads(line) for line in CORPUS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
ai = [row for row in rows if row["label"] == 1]
assert len(ai) == AI_ROWS, len(ai)
assert {row["generator"] for row in ai} == set(GENERATORS)
assert Counter(row["generator"] for row in ai) == Counter(EXPECTED_GENERATOR_COUNTS)
assert Counter(row["partition"] for row in ai) == Counter(EXPECTED_PARTITIONS)

print("corpus sha256 :", digest)
print("AI rows       :", len(ai))
print("generators    :", dict(Counter(row["generator"] for row in ai)))
print("partitions    :", dict(Counter(row["partition"] for row in ai)))

## 4. GPU / VRAM pre-flight and step estimate

In [ ]:
import math
import torch

assert torch.cuda.is_available(), (
    "CUDA is required. Runtime > Change runtime type > T4 GPU."
)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
print(f"GPU  : {props.name}")
print(f"VRAM : {vram_gb:.2f} GB (T4 class ~15.6 GB)")
assert props.total_memory >= 14.5e9, f"need >= 14.5 GB VRAM, found {vram_gb:.2f} GB"

all_micro = math.ceil(AI_ROWS / BATCH_SIZE)
all_steps_per_epoch = math.ceil(all_micro / GRAD_ACCUM)
all_total_steps = all_steps_per_epoch * EPOCHS
train_micro = math.ceil(TRAIN_AI_ROWS / BATCH_SIZE)
train_steps_per_epoch = math.ceil(train_micro / GRAD_ACCUM)
train_total_steps = train_steps_per_epoch * EPOCHS
# 11,887 AI rows / 16 = 743 micro-batches; / 2 = 372 steps/epoch; x3 = 1,116.
# The train partition: 9,344 / 16 = 584 micro-batches; 292/epoch; 876 total.
print(f"all AI rows  : {AI_ROWS} -> {all_steps_per_epoch} steps/epoch, {all_total_steps} total")
print(f"train AI rows: {TRAIN_AI_ROWS} -> {train_steps_per_epoch} steps/epoch, {train_total_steps} total")

## 5. Dependencies

In [ ]:
# Colab already ships a CUDA torch build; install the rest without replacing it.
%pip install -q "torch>=2.0" "transformers>=4.40" "accelerate>=0.30" "scikit-learn>=1.3"

import accelerate
import sklearn
import torch
import transformers

print("torch        :", torch.__version__)
print("cuda         :", torch.version.cuda)
print("transformers :", transformers.__version__)
print("accelerate   :", accelerate.__version__)
print("scikit-learn :", sklearn.__version__)

## 6. Train the three-class attribution model

In [ ]:
import subprocess, sys, time
from pathlib import Path

REPORT_DIR = Path(REPORT_DIR)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Exact run: python -m ml.train.train_attribution --device cuda --out /content/drive/MyDrive/StopTheCap_reports/E2C1_attribution
command = [
    sys.executable, "-m", "ml.train.train_attribution",
    "--corpus", str(REPO / "datasets/canonical/e2c1.jsonl"),
    "--out", str(REPORT_DIR),
    "--device", "cuda",
    "--max-length", str(MAX_LENGTH),
]
print("running:", " ".join(command), flush=True)
began = time.perf_counter()
completed = subprocess.run(command, cwd=str(REPO), text=True, capture_output=True)
wall_minutes = (time.perf_counter() - began) / 60
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(f"training failed with exit {completed.returncode}")
print(f"training wall time: {wall_minutes:.1f} min")

## 7. Print results and copy reports to Drive

In [ ]:
import json, shutil
from pathlib import Path

REPORT_DIR = Path(REPORT_DIR)
REPORT_PARENT = Path(REPORT_PARENT)
REPORT_PARENT.mkdir(parents=True, exist_ok=True)

result = json.loads((REPORT_DIR / "E2C1_attribution.json").read_text(encoding="utf-8"))
print("classes:", result["labels"])
print()
header = f"{'split':<13} {'n':>6} {'accuracy':>9} {'macro-F1':>9}"
print(header)
print("-" * len(header))
for name in ("val", "test_holdout"):
    report = result["splits"][name]
    print(f"{name:<13} {report['n']:>6} {report['accuracy']:>9.3f} {report['macro_f1']:>9.3f}")

print()
print("per-class (test_holdout):")
for label in result["labels"]:
    values = result["splits"]["test_holdout"]["per_class"][label]
    print(f"  {label:<20} support={values['support']:<5} "
          f"precision={values['precision']:.3f} recall={values['recall']:.3f} f1={values['f1']:.3f}")

print()
print("top confusions (test_holdout):")
for item in result["splits"]["test_holdout"]["top_confusions"]:
    print(f"  {item['true']} -> {item['predicted']}: {item['count']}")

ARTIFACTS = [
    "E2C1_attribution.json",
    "E2C1_attribution.md",
    "E2C1_attribution_predictions.jsonl",
]
for name in ARTIFACTS:
    source = REPORT_DIR / name
    assert source.is_file(), f"missing report: {source}"
    shutil.copy2(source, REPORT_PARENT / name)

checkpoint = REPORT_DIR / "checkpoints/last.pt"
print()
print("reports copied to:", REPORT_PARENT)
print("checkpoint       :", checkpoint, checkpoint.is_file())